In [ ]:
import os
from collections import defaultdict
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

import holidays
from dotenv import load_dotenv
from dalux_build import create_client

# Load environment variables from .env file
load_dotenv()

DALUX_BASE_URL = os.getenv("DALUX_BASE_URL")
DALUX_API_KEY = os.getenv("DALUX_API_KEY")

if not DALUX_BASE_URL or not DALUX_API_KEY:
    raise RuntimeError("Missing required environment variables. Check your .env file.")

# Create Dalux client
dalux = create_client(base_url=DALUX_BASE_URL, api_key=DALUX_API_KEY)
dk_holidays = holidays.DK()
copenhagen_tz = ZoneInfo("Europe/Copenhagen")

In [ ]:
projects = dalux.projects.list_projects()

In [ ]:
PROJECT_ID = os.getenv("PROJECT_ID")
if not PROJECT_ID:
    raise RuntimeError("Set PROJECT_ID in your environment.")

In [ ]:
project_tasks = dalux.tasks.get_project_tasks(
    PROJECT_ID
)
project_tasks.items

In [ ]:
TASK_TYPE_ID = os.getenv("TASK_TYPE_ID")
if not TASK_TYPE_ID:
    raise RuntimeError("Set TASK_TYPE_ID in your environment.")

tasks = dalux.tasks.get_all_project_tasks(PROJECT_ID, params={"typeId": TASK_TYPE_ID})
print(f"Retrieved {len(tasks)} tasks for project {PROJECT_ID}.")

In [ ]:
allowed_task_ids = {task.task_id for task in tasks}
all_task_changes = dalux.tasks.get_all_project_task_changes(PROJECT_ID)
tasks_changes = [change for change in all_task_changes if change.task_id in allowed_task_ids]
project_users_response = dalux.users.list_project_users(PROJECT_ID)
project_companies_response = dalux.companies.list_project_companies(PROJECT_ID)
work_packages_response = dalux.work_packages.list_work_packages(PROJECT_ID)

In [ ]:
project_users = project_users_response.items if project_users_response else []
project_companies = project_companies_response.items if project_companies_response else []
work_packages = work_packages_response.items if work_packages_response else []

In [ ]:
user_by_id = {user.user_id: user for user in project_users if user.user_id}
company_by_id = {company.company_id: company for company in project_companies if company.company_id}

In [ ]:
changes_by_task = defaultdict(list)
for change in tasks_changes:
    changes_by_task[change.task_id].append(change)

for task_id, thread in changes_by_task.items():
    thread.sort(key=lambda item: item.timestamp)

task_changes_thread = dict(changes_by_task)

tasks_with_ordered_changes = [
    {
        "task": task,
        "ordered_changes": task_changes_thread.get(task.task_id, []),
    }
    for task in tasks
]

print(f"Built ordered threads for {len(task_changes_thread)} tasks with changes.")
print(f"Attached ordered changes to {len(tasks_with_ordered_changes)} tasks.")

In [ ]:
len(tasks_with_ordered_changes)

In [ ]:
tasks_with_ordered_changes

In [ ]:
def get_modifier_user_id(change):
    if not change.fields or not change.fields.modified_by:
        return None
    return change.fields.modified_by.user_id


def to_copenhagen(dt):
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=ZoneInfo("UTC"))
    return dt.astimezone(copenhagen_tz)


def count_day_buckets(start_ts, end_ts):
    if end_ts <= start_ts:
        return {
            "total_days": 0,
            "weekend_holiday_days": 0,
            "business_days": 0,
        }

    start_local = to_copenhagen(start_ts)
    end_local = to_copenhagen(end_ts)

    if end_local.date() <= start_local.date():
        return {
            "total_days": 0,
            "weekend_holiday_days": 0,
            "business_days": 0,
        }

    current_day = start_local.date() + timedelta(days=1)
    end_day = end_local.date()
    total_days = 0
    weekend_holiday_days = 0
    business_days = 0

    while current_day <= end_day:
        total_days += 1
        is_weekday = current_day.weekday() < 5
        is_holiday = current_day in dk_holidays
        if is_weekday and not is_holiday:
            business_days += 1
        else:
            weekend_holiday_days += 1
        current_day += timedelta(days=1)

    return {
        "total_days": total_days,
        "weekend_holiday_days": weekend_holiday_days,
        "business_days": business_days,
    }


def business_days_between(start_ts, end_ts):
    return count_day_buckets(start_ts, end_ts)["business_days"]


def get_user_company_context(user_id):
    user = user_by_id.get(user_id) if user_id else None
    company_id = user.company_id if user else None
    company = company_by_id.get(company_id) if company_id else None
    return user, company


def get_user_display_name(user):
    if not user:
        return None
    full_name = " ".join(part for part in [user.first_name, user.last_name] if part).strip()
    return full_name or user.email

In [ ]:
detailed_transitions = []

for task_id, thread in task_changes_thread.items():
    for idx in range(1, len(thread)):
        prev_change = thread[idx - 1]
        next_change = thread[idx]

        from_user_id = get_modifier_user_id(prev_change)
        to_user_id = get_modifier_user_id(next_change)

        from_user, from_company = get_user_company_context(from_user_id)
        to_user, to_company = get_user_company_context(to_user_id)

        detailed_transitions.append(
            {
                "task_id": task_id,
                "from_timestamp": prev_change.timestamp,
                "to_timestamp": next_change.timestamp,
                "from_action": prev_change.action,
                "to_action": next_change.action,
                "from_user_name": get_user_display_name(from_user),
                "to_user_name": get_user_display_name(to_user),
                "from_user_email": getattr(from_user, "email", None),
                "to_user_email": getattr(to_user, "email", None),
                "from_company_id": getattr(from_user, "company_id", None),
                "to_company_id": getattr(to_user, "company_id", None),
                "from_company_name": getattr(from_company, "name", None),
                "to_company_name": getattr(to_company, "name", None),
                "response_days": business_days_between(prev_change.timestamp, next_change.timestamp),
            }
        )

print(f"Built {len(detailed_transitions)} transition rows.")

In [ ]:
EXCEPTION_COMPANY_IDS = {
# Add the company IDs of the exception companies here. 
}

CLOSING_ACTIONS = {"complete", "reject", "approve"} # Remove approve if you want to exclude approval actions from the closing actions.

def normalize_deadline_to_datetime(raw_deadline):
    if raw_deadline is None:
        return None

    if isinstance(raw_deadline, datetime):
        return raw_deadline

    if hasattr(raw_deadline, "year") and hasattr(raw_deadline, "month") and hasattr(raw_deadline, "day"):
        return datetime(
            raw_deadline.year,
            raw_deadline.month,
            raw_deadline.day,
            tzinfo=ZoneInfo("UTC"),
        )

    return None

exception_response_transitions = []

for task_id, thread in task_changes_thread.items():
    ordered_thread = sorted(thread, key=lambda item: item.timestamp)

    for idx, change in enumerate(ordered_thread):
        if change.action != "assign":
            continue

        assigned_to_user_id = getattr(getattr(change.fields, "current_responsible", None), "user_id", None) if change.fields else None
        assigned_to_user, assigned_to_company = get_user_company_context(assigned_to_user_id)
        assigned_company_id = getattr(assigned_to_user, "company_id", None)
        assigned_company_name = getattr(assigned_to_company, "name", None)

        if not assigned_company_id or assigned_company_id not in EXCEPTION_COMPANY_IDS:
            continue

        completion_event = None
        completion_idx = None
        for follow_idx, follow_change in enumerate(ordered_thread[idx + 1 :], start=idx + 1):
            if follow_change.action in CLOSING_ACTIONS:
                completion_event = follow_change
                completion_idx = follow_idx
                break

        if not completion_event:
            continue

        assign_modified_by = getattr(change.fields, "modified_by", None) if change.fields else None
        assign_modified_user_id = getattr(assign_modified_by, "user_id", None)
        assign_modified_user, assign_modified_company = get_user_company_context(assign_modified_user_id)

        completion_modified_by = getattr(completion_event.fields, "modified_by", None) if completion_event.fields else None
        completion_modified_user_id = getattr(completion_modified_by, "user_id", None)
        completion_modified_user, completion_modified_company = get_user_company_context(completion_modified_user_id)
        completion_company_id = getattr(completion_modified_user, "company_id", None)
        completion_company_name = getattr(completion_modified_company, "name", None)

        day_counts = count_day_buckets(change.timestamp, completion_event.timestamp)

        resolved_deadline_ts = None
        for deadline_change in reversed(ordered_thread[: completion_idx + 1]):
            deadline_value = normalize_deadline_to_datetime(
                getattr(deadline_change.fields, "deadline", None) if deadline_change.fields else None
            )
            if deadline_value is not None:
                resolved_deadline_ts = deadline_value
                break

        has_deadline = resolved_deadline_ts is not None
        if has_deadline:
            deadline_counts = count_day_buckets(resolved_deadline_ts, completion_event.timestamp)
            days_over_deadline = max(0, deadline_counts["business_days"])
            calendar_days_over_deadline = max(0, deadline_counts["total_days"])
        else:
            days_over_deadline = 0
            calendar_days_over_deadline = 0

        exception_response_transitions.append(
            {
                "task_id": task_id,
                "assign_timestamp": change.timestamp,
                "complete_timestamp": completion_event.timestamp,
                "assign_action": change.action,
                "complete_action": completion_event.action,
                "assigned_from_user_id": assign_modified_user_id,
                "assigned_from_user_name": get_user_display_name(assign_modified_user),
                "assigned_from_company_name": getattr(assign_modified_company, "name", None),
                "assigned_to_user_id": assigned_to_user_id,
                "assigned_to_user_name": get_user_display_name(assigned_to_user),
                "assigned_to_company_id": assigned_company_id,
                "assigned_to_company_name": assigned_company_name,
                "completed_by_user_id": completion_modified_user_id,
                "completed_by_user_name": get_user_display_name(completion_modified_user),
                "completed_by_company_id": completion_company_id,
                "completed_by_company_name": completion_company_name,
                "deadline_timestamp": resolved_deadline_ts,
                "has_deadline": has_deadline,
                "days_over_deadline": days_over_deadline,
                "calendar_days_over_deadline": calendar_days_over_deadline,
                "total_days": day_counts["total_days"],
                "weekend_holiday_days": day_counts["weekend_holiday_days"],
                "message_response_days": day_counts["business_days"],
            }
        )

print(f"Built {len(exception_response_transitions)} assign -> closing-action transitions for exception companies.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

UNASSIGNED_WORKPACKAGE_LABEL = "Unassigned"

task_created_ts_by_id = {}
for task_id, thread in task_changes_thread.items():
    if thread:
        task_created_ts_by_id[task_id] = min(change.timestamp for change in thread)

workpackage_by_id = {
    workpackage.workpackage_id: workpackage
    for workpackage in work_packages
    if getattr(workpackage, "workpackage_id", None)
}

task_workpackage_id_by_id = {}
task_workpackage_name_by_id = {}
for task_id, thread in task_changes_thread.items():
    resolved_workpackage_id = None
    for change in sorted(thread, key=lambda item: item.timestamp):
        if not change.fields:
            continue

        workpackage_id = getattr(change.fields, "workpackage_id", None)
        if workpackage_id:
            resolved_workpackage_id = workpackage_id
            break

    task_workpackage_id_by_id[task_id] = resolved_workpackage_id
    if not resolved_workpackage_id:
        task_workpackage_name_by_id[task_id] = UNASSIGNED_WORKPACKAGE_LABEL
        continue

    workpackage = workpackage_by_id.get(resolved_workpackage_id)
    workpackage_name = getattr(workpackage, "name", None) if workpackage else None
    task_workpackage_name_by_id[task_id] = workpackage_name or f"Unknown workpackage ({resolved_workpackage_id})"

task_number_by_id = {}
for task in tasks:
    task_number = getattr(task, "number", None)
    if task_number is None and isinstance(getattr(task, "data", None), dict):
        task_number = task.data.get("number")
    task_number_by_id[task.task_id] = task_number

exception_df = pd.DataFrame(exception_response_transitions).copy()
exception_df["task_number"] = exception_df["task_id"].map(task_number_by_id)
exception_df["task_created_ts"] = exception_df["task_id"].map(task_created_ts_by_id)
task_created_datetime = pd.to_datetime(exception_df["task_created_ts"])
exception_df["task_created_month"] = task_created_datetime.dt.strftime("%Y-%m")
exception_df["workpackage_name"] = exception_df["task_id"].map(task_workpackage_name_by_id).fillna(UNASSIGNED_WORKPACKAGE_LABEL)
exception_df["response_days_over_10"] = (exception_df["message_response_days"] - 10).clip(lower=0)
exception_df["response_days_over_5"] = (exception_df["message_response_days"] - 5).clip(lower=0)

print(f"Rows in exception_df: {len(exception_df)}")
print("Rows with task_number:", int(exception_df["task_number"].notna().sum()))
print("Resolved workpackage names:", int(exception_df["workpackage_name"].nunique(dropna=False)))
print(
    "Rows in Unassigned workpackage:",
    int((exception_df["workpackage_name"] == UNASSIGNED_WORKPACKAGE_LABEL).sum()),
)
exception_df[[
    "task_number",
    "workpackage_name",
    "assigned_from_user_name",
    "assigned_to_user_name",
    "completed_by_user_name",
    "assigned_to_company_name",
    "assign_timestamp",
    "complete_timestamp",
    "total_days",
    "message_response_days",
    "response_days_over_10",
]].head(10)

In [ ]:
task_days = (
    exception_df.groupby("task_number", as_index=False)["response_days_over_10"]
    .sum()
    .sort_values("response_days_over_10", ascending=False)
    .head(70)
)

plt.figure(figsize=(14, 6))
plt.bar(task_days["task_number"], task_days["response_days_over_10"], color="#2f5d8a")
plt.xticks(rotation=90)
plt.ylabel("Response days over 10")
plt.xlabel("Task Number")
plt.title("Top 50 Tasks af summen af dage over svartidsgrænsen på 10 dage")
plt.tight_layout()
plt.show()

In [ ]:
exception_df["response_days_over_10"].sum()

In [ ]:
exception_df["to_company_label"] = exception_df.apply(
    lambda row: f"{row['assigned_to_company_name']}" if pd.notna(row["assigned_to_company_name"]) else row["assigned_to_company_id"],
    axis=1,
)

company_order = sorted(exception_df["to_company_label"].dropna().unique().tolist())
company_data = [
    exception_df.loc[exception_df["to_company_label"] == company, "message_response_days"].tolist()
    for company in company_order
]

plt.figure(figsize=(12, 6))
plt.boxplot(company_data, tick_labels=company_order, showfliers=False)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Message response days")
plt.xlabel("Exception company")
plt.title("Message response days distribution by exception company")
plt.tight_layout()
plt.show()

total_over_10_days = float(exception_df["response_days_over_10"].sum())
total_over_5_days = float(exception_df["response_days_over_5"].sum())

print(f"Total message response days over legal 10 days: {total_over_10_days:.0f}")
print(f"Total message response days over 5 days: {total_over_5_days:.0f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from openpyxl import load_workbook
from openpyxl.chart import BarChart, Reference

UNASSIGNED_WORKPACKAGE_LABEL = "Unassigned"
UNASSIGNED_LABEL = "Unassigned"


task_created_ts_by_id = {
    task_id: min(change.timestamp for change in thread)
    for task_id, thread in task_changes_thread.items()
    if thread
}

workpackage_by_id = {
    workpackage.workpackage_id: workpackage
    for workpackage in work_packages
    if getattr(workpackage, "workpackage_id", None)
}

task_workpackage_id_by_id = {}
task_workpackage_name_by_id = {}
for task_id, thread in task_changes_thread.items():
    resolved_workpackage_id = next(
        (
            getattr(change.fields, "workpackage_id", None)
            for change in sorted(thread, key=lambda item: item.timestamp)
            if change.fields and getattr(change.fields, "workpackage_id", None)
        ),
        None,
    )
    task_workpackage_id_by_id[task_id] = resolved_workpackage_id
    if not resolved_workpackage_id:
        task_workpackage_name_by_id[task_id] = UNASSIGNED_WORKPACKAGE_LABEL
    else:
        workpackage = workpackage_by_id.get(resolved_workpackage_id)
        task_workpackage_name_by_id[task_id] = getattr(workpackage, "name", None) or f"Unknown workpackage ({resolved_workpackage_id})"

task_number_by_id = {}
for task in tasks:
    task_number = getattr(task, "number", None)
    if task_number is None and isinstance(getattr(task, "data", None), dict):
        task_number = task.data.get("number")
    task_number_by_id[task.task_id] = task_number

exception_df = pd.DataFrame(exception_response_transitions).copy()
exception_df["task_number"] = exception_df["task_id"].map(task_number_by_id)
exception_df["task_created_ts"] = exception_df["task_id"].map(task_created_ts_by_id)
exception_df["task_created_month"] = pd.to_datetime(exception_df["task_created_ts"]).dt.strftime("%Y-%m")
exception_df["workpackage_name"] = exception_df["task_id"].map(task_workpackage_name_by_id).fillna(UNASSIGNED_WORKPACKAGE_LABEL)
exception_df["assigned_to_user_name"] = exception_df["assigned_to_user_name"].fillna(UNASSIGNED_LABEL)
exception_df["assigned_to_company_name"] = exception_df["assigned_to_company_name"].fillna(UNASSIGNED_LABEL)
exception_df["has_deadline"] = exception_df["has_deadline"].fillna(False).astype(bool)
exception_df["days_over_deadline"] = pd.to_numeric(exception_df["days_over_deadline"], errors="coerce").fillna(0)
exception_df["calendar_days_over_deadline"] = pd.to_numeric(exception_df["calendar_days_over_deadline"], errors="coerce").fillna(0)
exception_df["business_days_without_holidays_weekends"] = exception_df["message_response_days"]

# Only count over-limit days for transitions where a deadline is configured.
exception_df["response_days_over_10"] = (
    (exception_df["business_days_without_holidays_weekends"] - 10).clip(lower=0)
    .where(exception_df["has_deadline"], 0)
    .astype(float)
)
exception_df["response_days_over_5"] = (
    (exception_df["business_days_without_holidays_weekends"] - 5).clip(lower=0)
    .where(exception_df["has_deadline"], 0)
    .astype(float)
)


task_level = (
    exception_df.groupby(["task_created_month", "workpackage_name", "task_id"], as_index=False)
    .agg(
        task_transition_count=("task_id", "size"),
        task_total_message_response_days=("message_response_days", "sum"),
        task_total_days_over_deadline=("days_over_deadline", "sum"),
        task_total_response_days_over_10=("response_days_over_10", "sum"),
        task_total_response_days_over_5=("response_days_over_5", "sum"),
    )
)

task_level["task_number"] = task_level["task_id"].map(task_number_by_id)

monthly_workpackage_delay = (
    task_level.groupby(["task_created_month", "workpackage_name"], as_index=False)
    .agg(
        transition_count=("task_transition_count", "sum"),
        distinct_task_count=("task_id", "nunique"),
        avg_task_message_response_days=("task_total_message_response_days", "mean"),
        avg_task_days_over_deadline=("task_total_days_over_deadline", "mean"),
        avg_task_response_days_over_10=("task_total_response_days_over_10", "mean"),
        avg_task_response_days_over_5=("task_total_response_days_over_5", "mean"),
        total_message_response_days=("task_total_message_response_days", "sum"),
        total_days_over_deadline=("task_total_days_over_deadline", "sum"),
        total_response_days_over_10=("task_total_response_days_over_10", "sum"),
        total_response_days_over_5=("task_total_response_days_over_5", "sum"),
    )
)

monthly_workpackage_delay["avg_task_message_response_days"] = monthly_workpackage_delay["avg_task_message_response_days"].round(0)
monthly_workpackage_delay["avg_task_days_over_deadline"] = monthly_workpackage_delay["avg_task_days_over_deadline"].round(0)
monthly_workpackage_delay["avg_task_response_days_over_10"] = monthly_workpackage_delay["avg_task_response_days_over_10"].round(0)
monthly_workpackage_delay["avg_task_response_days_over_5"] = monthly_workpackage_delay["avg_task_response_days_over_5"].round(0)
monthly_workpackage_delay = monthly_workpackage_delay.sort_values(
    ["task_created_month", "avg_task_response_days_over_10"], ascending=[True, False]
).reset_index(drop=True)


def pivot_matrix(df, index, columns, values):
    return (
        df.pivot(index=index, columns=columns, values=values)
        .sort_index()
        .sort_index(axis=1)
        .fillna(0)
    )


workpackage_month_avg_over_10_matrix = pivot_matrix(
    monthly_workpackage_delay,
    "workpackage_name",
    "task_created_month",
    "avg_task_response_days_over_10",
)
workpackage_month_total_over_10_matrix = pivot_matrix(
    monthly_workpackage_delay,
    "workpackage_name",
    "task_created_month",
    "total_response_days_over_10",
)
workpackage_month_avg_over_5_matrix = pivot_matrix(
    monthly_workpackage_delay,
    "workpackage_name",
    "task_created_month",
    "avg_task_response_days_over_5",
)
workpackage_month_total_over_5_matrix = pivot_matrix(
    monthly_workpackage_delay,
    "workpackage_name",
    "task_created_month",
    "total_response_days_over_5",
)
workpackage_month_distinct_task_count_matrix = pivot_matrix(
    monthly_workpackage_delay,
    "workpackage_name",
    "task_created_month",
    "distinct_task_count",
)

cumulative_workpackage_delay = (
    monthly_workpackage_delay.groupby("workpackage_name", dropna=False, as_index=False)
    .agg(
        month_count=("task_created_month", "nunique"),
        total_transition_count=("transition_count", "sum"),
        total_distinct_task_count=("distinct_task_count", "sum"),
        cumulative_avg_over_deadline_days=("avg_task_days_over_deadline", "sum"),
        cumulative_total_over_deadline_days=("total_days_over_deadline", "sum"),
        cumulative_avg_over_10_days=("avg_task_response_days_over_10", "sum"),
        cumulative_total_over_10_days=("total_response_days_over_10", "sum"),
        cumulative_avg_over_5_days=("avg_task_response_days_over_5", "sum"),
        cumulative_total_over_5_days=("total_response_days_over_5", "sum"),
        cumulative_total_message_response_days=("total_message_response_days", "sum"),
    )
)

cumulative_workpackage_delay["cumulative_avg_over_deadline_days"] = cumulative_workpackage_delay["cumulative_avg_over_deadline_days"].round(0)
cumulative_workpackage_delay["cumulative_avg_over_10_days"] = cumulative_workpackage_delay["cumulative_avg_over_10_days"].round(0)
cumulative_workpackage_delay["cumulative_avg_over_5_days"] = cumulative_workpackage_delay["cumulative_avg_over_5_days"].round(0)
cumulative_workpackage_delay = cumulative_workpackage_delay.sort_values(
    "cumulative_avg_over_10_days", ascending=False
).reset_index(drop=True)
overall_month_workpackage_average_over_10_days = float(monthly_workpackage_delay["avg_task_response_days_over_10"].sum())

monthly_transition_check = int(monthly_workpackage_delay["transition_count"].sum())
source_transition_count = int(len(exception_df))

output_dir = Path.cwd()
monthly_workpackage_path = output_dir / "monthly_workpackage_delay_summary.csv"
cumulative_workpackage_path = output_dir / "cumulative_workpackage_delay_summary.csv"
workpackage_over_10_path = output_dir / "workpackage_month_over_10_report.xlsx"
workpackage_over_5_path = output_dir / "workpackage_month_over_5_report.xlsx"

monthly_workpackage_delay.to_csv(monthly_workpackage_path, index=False)
cumulative_workpackage_delay.to_csv(cumulative_workpackage_path, index=False)


def to_excel_sheet_name(name, used_names):
    cleaned = str(name) if name is not None else UNASSIGNED_WORKPACKAGE_LABEL
    for char in "[]:*?/\\":
        cleaned = cleaned.replace(char, "_")
    cleaned = cleaned.strip() or UNASSIGNED_WORKPACKAGE_LABEL
    base_name = cleaned[:31]
    candidate = base_name
    counter = 1
    while candidate in used_names:
        suffix = f"_{counter}"
        candidate = f"{base_name[:31 - len(suffix)]}{suffix}"
        counter += 1
    used_names.add(candidate)
    return candidate


def write_report(file_path, limit_label, task_limit_column, row_limit_column, avg_matrix, total_matrix):
    task_list_df = task_level[
        ["task_number", "task_id", "task_created_month", "workpackage_name", "task_transition_count", task_limit_column]
    ].copy()
    task_list_df = task_list_df.rename(columns={task_limit_column: "days_over_limit"})
    task_list_df = task_list_df.sort_values(
        ["days_over_limit", "task_created_month", "workpackage_name", "task_number"],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)

    tasks_opened_per_month_df = (
        task_level.groupby("task_created_month", as_index=False)
        .agg(tasks_opened=("task_id", "size"))
        .sort_values("task_created_month")
        .reset_index(drop=True)
    )

    days_over_limit_by_user_df = (
        exception_df.groupby("assigned_to_user_name", dropna=False, as_index=False)
        .agg(
            task_count=("task_id", "nunique"),
            transition_count=("task_id", "size"),
            total_days_over_limit=(row_limit_column, "sum"),
        )
        .sort_values("total_days_over_limit", ascending=False)
        .reset_index(drop=True)
    )

    days_over_limit_by_company_df = (
        exception_df.groupby("assigned_to_company_name", dropna=False, as_index=False)
        .agg(
            task_count=("task_id", "nunique"),
            transition_count=("task_id", "size"),
            total_days_over_limit=(row_limit_column, "sum"),
        )
        .sort_values("total_days_over_limit", ascending=False)
        .reset_index(drop=True)
    )

    total_days_over_limit = float(task_list_df["days_over_limit"].sum())
    total_days_over_limit_with_deadline = float(
        exception_df.loc[exception_df["has_deadline"], row_limit_column].sum()
    )
    total_days_over_limit_without_deadline = float(
        exception_df.loc[~exception_df["has_deadline"], row_limit_column].sum()
    )

    summary_df = pd.DataFrame(
        [
            {"metric": "total_tasks", "value": int(task_list_df["task_id"].nunique())},
            {"metric": "tasks_with_delay", "value": int((task_list_df["days_over_limit"] > 0).sum())},
            {"metric": "total_days_over_limit", "value": total_days_over_limit},
            {"metric": "total_days_over_limit_with_deadline", "value": total_days_over_limit_with_deadline},
            {"metric": "total_days_over_limit_without_deadline", "value": total_days_over_limit_without_deadline},
            {"metric": f"cumulative_avg_over_{limit_label}_days", "value": float(monthly_workpackage_delay[f"avg_task_response_days_over_{limit_label}"].sum())}
        ]
    )

    used_sheet_names = set()
    with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="summary", index=False)
        task_list_df.to_excel(writer, sheet_name="task_list", index=False)
        days_over_limit_by_user_df.to_excel(writer, sheet_name="days_over_limit_by_user", index=False)
        days_over_limit_by_company_df.to_excel(writer, sheet_name="days_over_limit_by_company", index=False)
        tasks_opened_per_month_df.to_excel(writer, sheet_name="tasks_opened_per_month", index=False)
        avg_matrix.to_excel(writer, sheet_name="avg_task_over_limit")
        total_matrix.to_excel(writer, sheet_name="total_over_limit")
        workpackage_month_distinct_task_count_matrix.to_excel(writer, sheet_name="distinct_task_count")

        for workpackage_name in sorted(task_level["workpackage_name"].dropna().unique().tolist()):
            workpackage_rows = task_level.loc[
                task_level["workpackage_name"] == workpackage_name,
                ["task_number", "task_created_month", task_limit_column],
            ].copy()
            workpackage_rows["task_number"] = workpackage_rows["task_number"].fillna("Unknown task number")
            workpackage_task_matrix = (
                workpackage_rows.pivot_table(
                    index="task_number",
                    columns="task_created_month",
                    values=task_limit_column,
                    aggfunc="sum",
                    fill_value=0,
                )
                .sort_index()
                .sort_index(axis=1)
            )
            workpackage_task_matrix.to_excel(writer, sheet_name=to_excel_sheet_name(workpackage_name, used_sheet_names))

    workbook = load_workbook(file_path)

    workbook.save(file_path)


write_report(
    workpackage_over_10_path,
    "10",
    "task_total_response_days_over_10",
    "response_days_over_10",
    workpackage_month_avg_over_10_matrix,
    workpackage_month_total_over_10_matrix,
)

write_report(
    workpackage_over_5_path,
    "5",
    "task_total_response_days_over_5",
    "response_days_over_5",
    workpackage_month_avg_over_5_matrix,
    workpackage_month_total_over_5_matrix,
)

In [ ]:
top_workpackage_names = cumulative_workpackage_delay.head(12)["workpackage_name"].tolist()

monthly_avg_pivot = (
    monthly_workpackage_delay.pivot(
        index="task_created_month",
        columns="workpackage_name",
        values="avg_task_response_days_over_10",
    )
    .sort_index()
    .fillna(0)
    .sort_index(axis=1)
)

top_monthly_avg_pivot = monthly_avg_pivot.reindex(columns=top_workpackage_names).fillna(0)

fig, ax = plt.subplots(figsize=(14, 7))
x_labels = top_monthly_avg_pivot.index.astype(str)
bottom = pd.Series(0.0, index=top_monthly_avg_pivot.index)
colors = plt.cm.tab20.colors

for color_idx, workpackage_name in enumerate(top_monthly_avg_pivot.columns):
    values = top_monthly_avg_pivot[workpackage_name]
    ax.bar(
        x_labels,
        values,
        bottom=bottom,
        color=colors[color_idx % len(colors)],
        label=str(workpackage_name),
    )
    bottom = bottom + values

monthly_totals = top_monthly_avg_pivot.sum(axis=1)
for idx, total_value in enumerate(monthly_totals):
    if total_value > 0:
        ax.text(
            idx,
            total_value,
            f"{total_value:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=45, ha="right")
ax.set_ylabel("Average over-10 response days")
ax.set_xlabel("Task created month")
ax.set_title("Top workpackages by monthly average over-10 response days")
ax.legend(title="Workpackage", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

top_cumulative_workpackages = cumulative_workpackage_delay.head(20).copy()

plt.figure(figsize=(14, 7))
plt.bar(
    top_cumulative_workpackages["workpackage_name"].astype(str),
    top_cumulative_workpackages["cumulative_avg_over_10_days"],
    color="#1f6f8b",
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Cumulative average over-10 response days")
plt.xlabel("Workpackage")
plt.title("Top workpackages by cumulative summed monthly average over-10 response days")
plt.tight_layout()
plt.show()

print("Monthly average over-10 pivot for top cumulative workpackages:")
display(top_monthly_avg_pivot)

print("Top cumulative workpackages:")
display(top_cumulative_workpackages)

In [ ]:
monthly_deadline_totals = (
    monthly_workpackage_delay.groupby("task_created_month", as_index=False)
    .agg(
        total_over_10_days=("total_response_days_over_10", "sum"),
        total_over_5_days=("total_response_days_over_5", "sum"),
    )
    .sort_values("task_created_month")
    .reset_index(drop=True)
)

period_total_over_10 = int(monthly_deadline_totals["total_over_10_days"].sum())
period_total_over_5 = int(monthly_deadline_totals["total_over_5_days"].sum())

x_labels = monthly_deadline_totals["task_created_month"].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

bars_10 = axes[0].bar(x_labels, monthly_deadline_totals["total_over_10_days"], color="#d1495b")
axes[0].set_title(f"Total days over 10-day limit by month (all workpackages) | Period total: {period_total_over_10}")
axes[0].set_ylabel("Days over deadline")
axes[0].grid(axis="y", linestyle="--", alpha=0.35)
for bar in bars_10:
    height = bar.get_height()
    axes[0].annotate(
        f"{int(height)}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

bars_5 = axes[1].bar(x_labels, monthly_deadline_totals["total_over_5_days"], color="#2f6690")
axes[1].set_title(f"Total days over 5-day limit by month (all workpackages) | Period total: {period_total_over_5}")
axes[1].set_xlabel("Task created month")
axes[1].set_ylabel("Days over deadline")
axes[1].grid(axis="y", linestyle="--", alpha=0.35)
for bar in bars_5:
    height = bar.get_height()
    axes[1].annotate(
        f"{int(height)}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

display(monthly_deadline_totals)

In [ ]:
monthly_sum_of_workpackage_averages = (
    monthly_workpackage_delay.groupby("task_created_month", as_index=False)
    .agg(
        sum_avg_over_10_days=("avg_task_response_days_over_10", "sum"),
        sum_avg_over_5_days=("avg_task_response_days_over_5", "sum"),
    )
    .sort_values("task_created_month")
    .reset_index(drop=True)
)

period_sum_avg_over_10 = float(monthly_sum_of_workpackage_averages["sum_avg_over_10_days"].sum())
period_sum_avg_over_5 = float(monthly_sum_of_workpackage_averages["sum_avg_over_5_days"].sum())

x_labels = monthly_sum_of_workpackage_averages["task_created_month"].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

bars_10 = axes[0].bar(x_labels, monthly_sum_of_workpackage_averages["sum_avg_over_10_days"], color="#ef8354")
axes[0].set_title(f"Sum of workpackage monthly averages over 10-day limit | Period total: {period_sum_avg_over_10:.2f}")
axes[0].set_ylabel("Summed average days")
axes[0].grid(axis="y", linestyle="--", alpha=0.35)
for bar in bars_10:
    height = float(bar.get_height())
    axes[0].annotate(
        f"{height:.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

bars_5 = axes[1].bar(x_labels, monthly_sum_of_workpackage_averages["sum_avg_over_5_days"], color="#4f772d")
axes[1].set_title(f"Sum of workpackage monthly averages over 5-day limit | Period total: {period_sum_avg_over_5:.2f}")
axes[1].set_xlabel("Task created month")
axes[1].set_ylabel("Summed average days")
axes[1].grid(axis="y", linestyle="--", alpha=0.35)
for bar in bars_5:
    height = float(bar.get_height())
    axes[1].annotate(
        f"{height:.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

display(monthly_sum_of_workpackage_averages)

In [ ]:
monthly_business_over_deadline = (
    exception_df.loc[exception_df["has_deadline"]]
    .groupby("task_created_month", as_index=False)
    .agg(total_business_days_over_deadline=("days_over_deadline", "sum"))
    .sort_values("task_created_month")
    .reset_index(drop=True)
)

period_total_business_over_deadline = float(
    monthly_business_over_deadline["total_business_days_over_deadline"].sum()
    if not monthly_business_over_deadline.empty
    else 0.0
)

x_labels = monthly_business_over_deadline["task_created_month"].astype(str)

plt.figure(figsize=(14, 5))
bars = plt.bar(
    x_labels,
    monthly_business_over_deadline["total_business_days_over_deadline"],
    color="#3a7d44",
)
plt.title(
    f"Business days over actual deadline by task created month | Period total: {period_total_business_over_deadline:.0f}"
)
plt.xlabel("Task created month")
plt.ylabel("Business days over deadline")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.35)

for bar in bars:
    height = float(bar.get_height())
    plt.annotate(
        f"{height:.0f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

display(monthly_business_over_deadline)

In [ ]:
monthly_sum_of_workpackage_deadline_averages = (
    monthly_workpackage_delay.groupby("task_created_month", as_index=False)
    .agg(sum_avg_days_over_deadline=("avg_task_days_over_deadline", "sum"))
    .sort_values("task_created_month")
    .reset_index(drop=True)
)

period_sum_avg_over_deadline = float(
    monthly_sum_of_workpackage_deadline_averages["sum_avg_days_over_deadline"].sum()
    if not monthly_sum_of_workpackage_deadline_averages.empty
    else 0.0
)

x_labels = monthly_sum_of_workpackage_deadline_averages["task_created_month"].astype(str)

plt.figure(figsize=(14, 5))
bars = plt.bar(
    x_labels,
    monthly_sum_of_workpackage_deadline_averages["sum_avg_days_over_deadline"],
    color="#bc6c25",
)
plt.title(
    f"Sum of workpackage monthly averages over deadline (business days) | Period total: {period_sum_avg_over_deadline:.2f}"
)
plt.xlabel("Task created month")
plt.ylabel("Summed average business days over deadline")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.35)

for bar in bars:
    height = float(bar.get_height())
    plt.annotate(
        f"{height:.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

display(monthly_sum_of_workpackage_deadline_averages)

In [ ]:
tasks_with_deadline = (
    exception_df.loc[exception_df["has_deadline"]].copy()
    .sort_values(["task_id", "assign_timestamp"])
    .reset_index(drop=True)
)

# One row per task: use first observed deadline tied to the task's creation month.
task_created_to_deadline = (
    tasks_with_deadline.groupby("task_id", as_index=False)
    .agg(
        task_created_ts=("task_created_ts", "first"),
        task_created_month=("task_created_month", "first"),
        deadline_timestamp=("deadline_timestamp", "first"),
    )
)

task_created_to_deadline["created_to_deadline_business_days"] = task_created_to_deadline.apply(
    lambda row: business_days_between(
        pd.to_datetime(row["task_created_ts"]).to_pydatetime(),
        pd.to_datetime(row["deadline_timestamp"]).to_pydatetime(),
    ),
    axis=1,
 )

monthly_created_to_deadline_avg = (
    task_created_to_deadline.groupby("task_created_month", as_index=False)
    .agg(
        task_count=("task_id", "nunique"),
        avg_business_days_created_to_deadline=("created_to_deadline_business_days", "mean"),
    )
    .sort_values("task_created_month")
    .reset_index(drop=True)
)
monthly_created_to_deadline_avg["avg_business_days_created_to_deadline"] = monthly_created_to_deadline_avg[
    "avg_business_days_created_to_deadline"
].round(2)

overall_avg_business_days_created_to_deadline = float(
    task_created_to_deadline["created_to_deadline_business_days"].mean()
    if not task_created_to_deadline.empty
    else 0.0
)

plt.figure(figsize=(14, 5))
bars = plt.bar(
    monthly_created_to_deadline_avg["task_created_month"].astype(str),
    monthly_created_to_deadline_avg["avg_business_days_created_to_deadline"],
    color="#6c757d",
)
plt.title(
    f"Average business days from task creation to deadline by month | Overall avg: {overall_avg_business_days_created_to_deadline:.2f}"
)
plt.xlabel("Task created month")
plt.ylabel("Average business days (created -> deadline)")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.35)

for bar in bars:
    height = float(bar.get_height())
    plt.annotate(
        f"{height:.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

display(monthly_created_to_deadline_avg)

In [ ]:
import html
import re
from collections import defaultdict
from datetime import datetime, time, timedelta
from zoneinfo import ZoneInfo

import holidays
import pandas as pd
import plotly.graph_objects as go

TIMELINE_TIMEZONE = ZoneInfo("Europe/Copenhagen")
ACTIVE_LIMIT_BUSINESS_DAYS = 10

# Override this if your Dalux tenant uses a different deep-link format.
# Available placeholders: {base_url}, {project_id}, {task_id}
DALUX_TASK_URL_TEMPLATE = "{base_url}/project/{project_id}/tasks/{task_id}"

timeline_task_numbers = None  # Example: ["TASK-1", "TASK-8", "TASK-293"]
timeline_created_from = None  # Example: "2025-07-01"
timeline_created_to = None  # Example: "2025-12-31"
timeline_only_exception_tasks = False  # Set True to hide tasks without exception transitions.


def _to_local_timestamp(value):
    if value is None:
        return None

    ts = pd.Timestamp(value)
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    return ts.tz_convert(TIMELINE_TIMEZONE)


def _deadline_to_timestamp(value):
    if value is None:
        return None

    deadline_ts = pd.Timestamp(datetime.combine(value, time(12, 0)))
    return deadline_ts.tz_localize(TIMELINE_TIMEZONE)


def _task_number_sort_key(task_number):
    match = re.search(r"(\d+)", str(task_number))
    return int(match.group(1)) if match else float("inf")


def _coerce_task_record(item):
    if isinstance(item, dict):
        task = item.get("task")
        ordered_changes = item.get("ordered_changes", [])
    else:
        task = getattr(item, "task", item)
        ordered_changes = getattr(item, "ordered_changes", [])

    return task, list(ordered_changes or [])


def _get_attr_or_key(item, name, default=None):
    if isinstance(item, dict):
        return item.get(name, default)
    return getattr(item, name, default)


def _actor_label(actor):
    if actor is None:
        return None

    if isinstance(actor, dict):
        for key in ("name", "user_name", "role_name", "email", "user_id", "role_id"):
            value = actor.get(key)
            if value:
                return str(value)
        return None

    for attr in ("name", "user_name", "role_name", "email", "user_id", "role_id"):
        value = getattr(actor, attr, None)
        if value:
            return str(value)
    return None


def _clean_text(value, fallback=""):
    if value is None:
        return fallback
    text = str(value).strip()
    return text if text else fallback


def _hover_lines(*lines):
    return "<br>".join(html.escape(str(line)) for line in lines if line is not None and str(line) != "")


def _format_ts(ts):
    if ts is None:
        return ""
    return _to_local_timestamp(ts).strftime("%Y-%m-%d %H:%M")


def _plot_x(ts):
    return _to_local_timestamp(ts).to_pydatetime().replace(tzinfo=None)


def _task_url(task_id):
    if not task_id:
        return None

    template = str(globals().get("DALUX_TASK_URL_TEMPLATE") or "").strip()
    if not template:
        return None

    base_url = str(globals().get("DALUX_BASE_URL") or "").rstrip("/")
    project_id = str(globals().get("PROJECT_ID") or "")
    try:
        return template.format(base_url=base_url, project_id=project_id, task_id=str(task_id))
    except Exception:
        return None


def _task_link_label(task_id, task_number):
    safe_label = html.escape(str(task_number))
    url = _task_url(task_id)
    if not url:
        return safe_label
    return f'<a href="{html.escape(url, quote=True)}" target="_blank">{safe_label}</a>'


def _event_hover(record, event):
    lines = [
        f"{record['task_number']} - {event['kind'].title()}",
        f"Time: {_format_ts(event['timestamp'])}",
    ]
    if event.get("assigned_to"):
        lines.append(f"Assigned to: {event['assigned_to']}")
    if event.get("modified_by"):
        lines.append(f"Modified by: {event['modified_by']}")
    if event.get("description"):
        lines.append(_clean_text(event["description"])[:240])
    return _hover_lines(*lines)


def _history_segment_hover(record, from_event, to_event):
    return _hover_lines(
        f"{record['task_number']} - history segment",
        f"From: {_format_ts(from_event['timestamp'])} ({from_event['kind']})",
        f"To: {_format_ts(to_event['timestamp'])} ({to_event['kind']})",
        f"Modified by: {to_event.get('modified_by') or 'Unknown'}",
    )


def _build_all_active_intervals(transitions):
    intervals_by_task_id = defaultdict(list)

    for transition in transitions:
        task_id = _get_attr_or_key(transition, "task_id")
        start_ts = _to_local_timestamp(_get_attr_or_key(transition, "assign_timestamp"))
        end_ts = _to_local_timestamp(_get_attr_or_key(transition, "complete_timestamp"))
        is_exception = _get_attr_or_key(transition, "is_exception", False)
        if not task_id or start_ts is None or end_ts is None or end_ts <= start_ts:
            continue

        intervals_by_task_id[str(task_id)].append(
            {
                "start": start_ts,
                "end": end_ts,
                "start_action": _get_attr_or_key(transition, "assign_action", "assign"),
                "end_action": _get_attr_or_key(transition, "complete_action", "closing action"),
                "assigned_from": _get_attr_or_key(transition, "assigned_from_user_name"),
                "assigned_from_company": _get_attr_or_key(transition, "assigned_from_company_name"),
                "assigned_to": _get_attr_or_key(transition, "assigned_to_user_name"),
                "assigned_to_company": _get_attr_or_key(transition, "assigned_to_company_name"),
                "completed_by": _get_attr_or_key(transition, "completed_by_user_name"),
                "completed_by_company": _get_attr_or_key(transition, "completed_by_company_name"),
                "business_days": _get_attr_or_key(transition, "message_response_days"),
                "is_exception": is_exception,
            }
        )

    return {
        task_id: sorted(intervals, key=lambda item: item["start"])
        for task_id, intervals in intervals_by_task_id.items()
    }


def _build_timeline_records(task_items, active_intervals_by_task_id):
    records = []

    for item in task_items:
        task, ordered_changes = _coerce_task_record(item)
        if task is None:
            continue

        task_number = getattr(task, "number", None)
        task_created = getattr(task, "created", None)
        task_id = getattr(task, "task_id", None)
        task_title = getattr(task, "title", None)

        if isinstance(task, dict):
            task_number = task_number or task.get("number")
            task_created = task_created or task.get("created")
            task_id = task_id or task.get("task_id") or task.get("taskId")
            task_title = task_title or task.get("title")

        task_data = getattr(task, "data", None)
        if isinstance(task_data, dict):
            task_number = task_number or task_data.get("number")
            task_title = task_title or task_data.get("title")

        if task_number is None:
            continue

        ordered_changes = sorted(
            ordered_changes,
            key=lambda change: getattr(change, "timestamp", pd.Timestamp.min),
        )

        if task_created is None:
            for change in ordered_changes:
                change_ts = getattr(change, "timestamp", None)
                if change_ts is not None:
                    task_created = change_ts
                    break

        created_ts = _to_local_timestamp(task_created)
        if created_ts is None:
            continue

        deadline_ts = None
        events = [
            {
                "timestamp": created_ts,
                "kind": "created",
                "description": task_title,
                "assigned_to": None,
                "modified_by": None,
            }
        ]

        for change in ordered_changes:
            timestamp = _to_local_timestamp(getattr(change, "timestamp", None))
            if timestamp is None:
                continue

            action = str(getattr(change, "action", "other") or "other").lower()
            if action not in {"assign", "update", "complete", "approve", "reject"}:
                action = "other"

            fields = getattr(change, "fields", None)
            deadline_value = getattr(fields, "deadline", None) if fields else None
            if deadline_value is not None and deadline_ts is None:
                deadline_ts = _deadline_to_timestamp(deadline_value)

            events.append(
                {
                    "timestamp": timestamp,
                    "kind": action,
                    "description": getattr(change, "description", None),
                    "assigned_to": _actor_label(getattr(fields, "assigned_to", None) if fields else None),
                    "modified_by": _actor_label(getattr(fields, "modified_by", None) if fields else None),
                }
            )

        events = sorted(events, key=lambda event: event["timestamp"])
        active_intervals = active_intervals_by_task_id.get(str(task_id), [])

        records.append(
            {
                "task_id": task_id,
                "task_number": str(task_number),
                "title": task_title,
                "created_ts": created_ts,
                "deadline_ts": deadline_ts,
                "events": events,
                "active_intervals": active_intervals,
            }
        )

    return records


def _date_bounds(records):
    timestamps = []
    for record in records:
        timestamps.extend(event["timestamp"] for event in record["events"])
        timestamps.extend(interval["start"] for interval in record["active_intervals"])
        timestamps.extend(interval["end"] for interval in record["active_intervals"])
        if record["deadline_ts"] is not None:
            timestamps.append(record["deadline_ts"])
    return min(timestamps), max(timestamps)


def _danish_holiday_dates(start_ts, end_ts):
    years = range(start_ts.year - 1, end_ts.year + 2)
    holiday_calendar = holidays.DK(years=years)
    return {holiday_date for holiday_date in holiday_calendar if holiday_date.weekday() < 5}


def _is_business_date(day, holiday_dates):
    return day.weekday() < 5 and day not in holiday_dates


def _first_over_limit_timestamp(start_ts, limit_business_days, holiday_dates):
    # Matches count_day_buckets(): the start date is excluded, and the end date is counted.
    current_day = _to_local_timestamp(start_ts).date()
    business_days_seen = 0

    while business_days_seen <= limit_business_days:
        current_day += timedelta(days=1)
        if _is_business_date(current_day, holiday_dates):
            business_days_seen += 1

    return pd.Timestamp(datetime.combine(current_day, time.min)).tz_localize(TIMELINE_TIMEZONE)


def _queue_line(bucket, start_ts, end_ts, y_pos, hover_text):
    if end_ts is None or start_ts is None or end_ts <= start_ts:
        return
    bucket["x"].extend([_plot_x(start_ts), _plot_x(end_ts), None])
    bucket["y"].extend([y_pos, y_pos, None])
    bucket["text"].extend([hover_text, hover_text, None])


def _queue_marker(bucket, ts, y_pos, hover_text):
    if ts is None:
        return
    bucket["x"].append(_plot_x(ts))
    bucket["y"].append(y_pos)
    bucket["text"].append(hover_text)


def _interval_hover(record, interval, segment_label):
    lines = [
        f"{record['task_number']} - {segment_label}",
        f"Start: {_format_ts(interval['start'])}",
        f"End: {_format_ts(interval['end'])}",
        f"Closing action: {interval['end_action']}",
    ]
    if interval.get("business_days") is not None:
        lines.append(f"Business days: {interval['business_days']}")
    if interval.get("assigned_to") or interval.get("assigned_to_company"):
        lines.append(
            "Assigned to: "
            + " / ".join(
                item for item in [interval.get("assigned_to"), interval.get("assigned_to_company")] if item
            )
        )
    if interval.get("completed_by") or interval.get("completed_by_company"):
        lines.append(
            "Closed by: "
            + " / ".join(
                item for item in [interval.get("completed_by"), interval.get("completed_by_company")] if item
            )
        )
    return _hover_lines(*lines)


# Build ALL transitions (from full task history), then classify exception ones.
exception_company_ids = globals().get("EXCEPTION_COMPANY_IDS", set())
all_transitions = []

for task_id, thread in task_changes_thread.items():
    ordered_thread = sorted(thread, key=lambda item: item.timestamp)

    for idx, change in enumerate(ordered_thread):
        if change.action != "assign":
            continue

        assigned_to_user_id = getattr(getattr(change.fields, "current_responsible", None), "user_id", None) if change.fields else None
        assigned_to_user, assigned_to_company = get_user_company_context(assigned_to_user_id)
        assigned_company_id = getattr(assigned_to_user, "company_id", None)
        assigned_company_name = getattr(assigned_to_company, "name", None)

        completion_event = None
        completion_idx = None
        for follow_idx, follow_change in enumerate(ordered_thread[idx + 1 :], start=idx + 1):
            if follow_change.action in globals().get("CLOSING_ACTIONS", {"complete", "reject", "approve"}):
                completion_event = follow_change
                completion_idx = follow_idx
                break

        if not completion_event:
            continue

        assign_modified_by = getattr(change.fields, "modified_by", None) if change.fields else None
        assign_modified_user_id = getattr(assign_modified_by, "user_id", None)
        assign_modified_user, assign_modified_company = get_user_company_context(assign_modified_user_id)

        completion_modified_by = getattr(completion_event.fields, "modified_by", None) if completion_event.fields else None
        completion_modified_user_id = getattr(completion_modified_by, "user_id", None)
        completion_modified_user, completion_modified_company = get_user_company_context(completion_modified_user_id)
        completion_company_id = getattr(completion_modified_user, "company_id", None)
        completion_company_name = getattr(completion_modified_company, "name", None)

        day_counts = count_day_buckets(change.timestamp, completion_event.timestamp)

        resolved_deadline_ts = None
        for deadline_change in reversed(ordered_thread[: completion_idx + 1]):
            deadline_value = normalize_deadline_to_datetime(
                getattr(deadline_change.fields, "deadline", None) if deadline_change.fields else None
            )
            if deadline_value is not None:
                resolved_deadline_ts = deadline_value
                break

        has_deadline = resolved_deadline_ts is not None
        if has_deadline:
            deadline_counts = count_day_buckets(resolved_deadline_ts, completion_event.timestamp)
            days_over_deadline = max(0, deadline_counts["business_days"])
            calendar_days_over_deadline = max(0, deadline_counts["total_days"])
        else:
            days_over_deadline = 0
            calendar_days_over_deadline = 0

        is_exception = bool(assigned_company_id and assigned_company_id in exception_company_ids)

        all_transitions.append(
            {
                "task_id": task_id,
                "assign_timestamp": change.timestamp,
                "complete_timestamp": completion_event.timestamp,
                "assign_action": change.action,
                "complete_action": completion_event.action,
                "assigned_from_user_id": assign_modified_user_id,
                "assigned_from_user_name": get_user_display_name(assign_modified_user),
                "assigned_from_company_name": getattr(assign_modified_company, "name", None),
                "assigned_to_user_id": assigned_to_user_id,
                "assigned_to_user_name": get_user_display_name(assigned_to_user),
                "assigned_to_company_id": assigned_company_id,
                "assigned_to_company_name": assigned_company_name,
                "completed_by_user_id": completion_modified_user_id,
                "completed_by_user_name": get_user_display_name(completion_modified_user),
                "completed_by_company_id": completion_company_id,
                "completed_by_company_name": completion_company_name,
                "deadline_timestamp": resolved_deadline_ts,
                "has_deadline": has_deadline,
                "days_over_deadline": days_over_deadline,
                "calendar_days_over_deadline": calendar_days_over_deadline,
                "total_days": day_counts["total_days"],
                "weekend_holiday_days": day_counts["weekend_holiday_days"],
                "message_response_days": day_counts["business_days"],
                "is_exception": is_exception,
            }
        )

print(f"Built {len(all_transitions)} assign -> closing-action transitions from full task history.")

active_intervals_by_task_id = _build_all_active_intervals(all_transitions)
timeline_records = _build_timeline_records(tasks_with_ordered_changes, active_intervals_by_task_id)

if timeline_task_numbers:
    selected_task_numbers = {str(item).strip().upper() for item in timeline_task_numbers}
    timeline_records = [
        record
        for record in timeline_records
        if record["task_number"].upper() in selected_task_numbers
    ]

if timeline_created_from:
    created_from_ts = pd.Timestamp(timeline_created_from).tz_localize(TIMELINE_TIMEZONE)
    timeline_records = [
        record for record in timeline_records if record["created_ts"] >= created_from_ts
    ]

if timeline_created_to:
    created_to_ts = pd.Timestamp(timeline_created_to).tz_localize(TIMELINE_TIMEZONE) + pd.Timedelta(days=1)
    timeline_records = [
        record for record in timeline_records if record["created_ts"] < created_to_ts
    ]

if timeline_only_exception_tasks:
    timeline_records = [
        record
        for record in timeline_records
        if any(interval.get("is_exception") for interval in record["active_intervals"])
    ]

timeline_records = sorted(
    timeline_records,
    key=lambda record: (_task_number_sort_key(record["task_number"]), record["created_ts"]),
)

if not timeline_records:
    raise ValueError("No tasks matched the selected timeline filters.")

start_bound, end_bound = _date_bounds(timeline_records)
holiday_dates = _danish_holiday_dates(start_bound, end_bound)
holiday_values = sorted(day.isoformat() for day in holiday_dates)

line_buckets = {
    "history_dashed": {"x": [], "y": [], "text": []},
    "exception_on_time": {"x": [], "y": [], "text": []},
    "exception_over_10": {"x": [], "y": [], "text": []},
}
marker_buckets = {
    "created": {"x": [], "y": [], "text": []},
    "deadline": {"x": [], "y": [], "text": []},
    "assign": {"x": [], "y": [], "text": []},
    "complete": {"x": [], "y": [], "text": []},
    "approve": {"x": [], "y": [], "text": []},
    "reject": {"x": [], "y": [], "text": []},
}

for y_pos, record in enumerate(timeline_records):
    events = record["events"]

    for idx in range(1, len(events)):
        _queue_line(
            line_buckets["history_dashed"],
            events[idx - 1]["timestamp"],
            events[idx]["timestamp"],
            y_pos,
            _history_segment_hover(record, events[idx - 1], events[idx]),
        )

    _queue_marker(marker_buckets["created"], record["created_ts"], y_pos, _event_hover(record, events[0]))

    if record["deadline_ts"] is not None:
        deadline_hover = _hover_lines(
            f"{record['task_number']} - Deadline",
            f"Date: {_format_ts(record['deadline_ts'])}",
        )
        _queue_marker(marker_buckets["deadline"], record["deadline_ts"], y_pos, deadline_hover)

    for event in events[1:]:
        if event["kind"] in marker_buckets:
            _queue_marker(marker_buckets[event["kind"]], event["timestamp"], y_pos, _event_hover(record, event))

    for interval in record["active_intervals"]:
        if not interval.get("is_exception"):
            continue

        over_limit_start = _first_over_limit_timestamp(
            interval["start"], ACTIVE_LIMIT_BUSINESS_DAYS, holiday_dates
        )

        on_time_end = min(interval["end"], over_limit_start)
        _queue_line(
            line_buckets["exception_on_time"],
            interval["start"],
            on_time_end,
            y_pos,
            _interval_hover(record, interval, f"Exception exchange <= {ACTIVE_LIMIT_BUSINESS_DAYS} business days"),
        )

        if interval["end"] > over_limit_start:
            _queue_line(
                line_buckets["exception_over_10"],
                max(interval["start"], over_limit_start),
                interval["end"],
                y_pos,
                _interval_hover(record, interval, f"Exception exchange > {ACTIVE_LIMIT_BUSINESS_DAYS} business days"),
            )

fig = go.Figure()

line_styles = {
    "history_dashed": {
        "name": "Full task history (dashed)",
        "color": "#6b7280",
        "width": 2,
        "dash": "dot",
    },
    "exception_on_time": {
        "name": f"Exception exchanges <= {ACTIVE_LIMIT_BUSINESS_DAYS} business days",
        "color": "#2563eb",
        "width": 5,
        "dash": "solid",
    },
    "exception_over_10": {
        "name": f"Exception exchanges > {ACTIVE_LIMIT_BUSINESS_DAYS} business days",
        "color": "#dc2626",
        "width": 5,
        "dash": "solid",
    },
}

# Trace order matters: dashed history at the back, blue above it, red above blue.
for key in ["history_dashed", "exception_on_time", "exception_over_10"]:
    bucket = line_buckets[key]
    if not bucket["x"]:
        continue
    style = line_styles[key]
    fig.add_trace(
        go.Scatter(
            x=bucket["x"],
            y=bucket["y"],
            mode="lines",
            name=style["name"],
            line={"color": style["color"], "width": style["width"], "dash": style["dash"]},
            text=bucket["text"],
            hovertemplate="%{text}<extra></extra>",
        )
    )

marker_styles = {
    "created": {"name": "Created", "symbol": "circle", "color": "#111827", "size": 8},
    "deadline": {"name": "Deadline", "symbol": "diamond-open", "color": "#f59e0b", "size": 9},
    "assign": {"name": "Assign", "symbol": "square", "color": "#7c3aed", "size": 8},
    "complete": {"name": "Complete", "symbol": "triangle-down", "color": "#0f766e", "size": 9},
    "approve": {"name": "Approve", "symbol": "triangle-up", "color": "#16a34a", "size": 10},
    "reject": {"name": "Reject", "symbol": "x", "color": "#dc2626", "size": 10},
}

for key in ["created", "deadline", "assign", "complete", "approve", "reject"]:
    bucket = marker_buckets[key]
    if not bucket["x"]:
        continue
    style = marker_styles[key]
    fig.add_trace(
        go.Scatter(
            x=bucket["x"],
            y=bucket["y"],
            mode="markers",
            name=style["name"],
            marker={
                "symbol": style["symbol"],
                "color": style["color"],
                "size": style["size"],
                "line": {"width": 1.5, "color": style["color"]},
            },
            text=bucket["text"],
            hovertemplate="%{text}<extra></extra>",
        )
    )

# Add thin horizontal separator lines between tasks
for separator_y in [i + 0.5 for i in range(len(timeline_records) - 1)]:
    fig.add_hline(
        y=separator_y,
        line_dash="solid",
        line_color="#d1d5db",
        line_width=0.5,
        layer="below",
    )

# Add very thin daily vertical lines
day_cursor = pd.Timestamp(start_bound.date()).tz_localize(TIMELINE_TIMEZONE)
end_day = pd.Timestamp(end_bound.date()).tz_localize(TIMELINE_TIMEZONE)
while day_cursor <= end_day:
    fig.add_vline(
        x=_plot_x(day_cursor),
        line_dash="solid",
        line_color="#e5e7eb",
        line_width=0.25,
        layer="below",
    )
    day_cursor += pd.Timedelta(days=1)

# Create clickable y-axis side labels using annotations
task_link_annotations = [
    {
        "xref": "paper",
        "yref": "y",
        "x": -0.03,
        "y": idx,
        "text": _task_link_label(record.get("task_id"), record.get("task_number")),
        "showarrow": False,
        "xanchor": "right",
        "align": "right",
        "font": {"size": 11, "color": "#1f2937"},
    }
    for idx, record in enumerate(timeline_records)
]

fig_height = min(1800, max(700, 20 * len(timeline_records) + 220))
exception_interval_count = sum(
    1
    for record in timeline_records
    for interval in record["active_intervals"]
    if interval.get("is_exception")
)

fig.update_layout(
    title=(
        "Task timeline - full history with exception response windows"
        f" ({len(timeline_records)} tasks, {exception_interval_count} exception exchanges)"
    ),
    template="plotly_white",
    autosize=True,
    height=fig_height,
    hovermode="closest",
    dragmode="zoom",
    margin={"l": 220, "r": 40, "t": 80, "b": 80},
    legend={"orientation": "v", "yanchor": "top", "y": 0.99, "xanchor": "left", "x": 1.01},
    annotations=task_link_annotations,
)

fig.update_xaxes(
    title="Business days (weekends and Danish public holidays removed)",
    rangebreaks=[
        {"bounds": ["sat", "mon"]},
        {"values": holiday_values},
    ],
    rangeslider={"visible": True, "thickness": 0.06},
    tickformat="%Y-%m-%d",
    showgrid=True,
    gridcolor="#e5e7eb",
)
fig.update_yaxes(
    title="Task number (click label to open task)",
    tickmode="array",
    tickvals=list(range(len(timeline_records))),
    ticktext=[record["task_number"] for record in timeline_records],
    showticklabels=False,
    autorange="reversed",
    fixedrange=False,
    scaleanchor=None,
    showgrid=False,
)

fig.show(renderer="browser", config={"responsive": True})

print(
    f"Plotted {len(timeline_records)} tasks with dashed full-history segments from creation to end, "
    f"{exception_interval_count} exception exchanges (blue on-time, red over-10), "
    "and thin daily grid lines."
)
print(
    "Each task label on the left is a clickable Dalux link. "
    "If links do not open correctly, adjust DALUX_TASK_URL_TEMPLATE at the top of this cell."
)

In [ ]:
fig.write_html("dalux_task_timeline.html", include_plotlyjs="cdn", full_html=True)